# 🏭 Industrial Predictive Maintenance & Fault Diagnostic Engine

---

# Phase 2: Exploratory Data Analysis (EDA)

After **framing the problem and data** in Phase 1, we now move to the **exploration** phase. The goal is not decoration, but **evidence-based engineering decisions**:

- What is the distribution of each sensor? Are there outliers?
- How do readings differ between **healthy** and **failed** machines? (this is the separation the model will learn)
- Which sensors are **most correlated** with failure? (will guide feature engineering later)
- What is the relationship between **product quality (Type)** and failure rate?

## 🎨 Plotting tools used

| Library | Usage |
|---------|-----------|
| `matplotlib` | The base for programmatic plotting in Python |
| `seaborn` | An elegant layer over matplotlib with ready statistical plots (histplot, boxplot, heatmap) |

**Why do we care about plot quality?** Because visual exploration reveals patterns that raw numbers do not show.


## 2.0 — Setup: imports + rebuilding the data (shortcut from Phase 1)

To make this notebook **self-contained**, we reload and build the targets with a shortened version of Phase 1. In your final project you will unify this logic into a single `utils.py` file (we will do that in Phase 6).


In [ ]:
# ============================================================
# Phase 2 — Setup
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ─── Constants (same as Phase 1) ────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_FILE = Path("/kaggle/input/ai4i-predictive-maintenance-dataset/ai4i2020.csv")

SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
BINARY_TARGET = "Machine failure"
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
MULTI_CLASS_TARGET = "Failure Type"
CATEGORICAL_COL = "Type"   # product quality: Low / Medium / High

# ─── Plot settings (uniform, clean style) ──────────────────
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110          # higher image resolution
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

# ─── Load data ──────────────────────────────────────────────
df = pd.read_csv(DATA_FILE)

# ─── Build multi-class target (same policy as Phase 1) ──────
flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)
df[MULTI_CLASS_TARGET] = "No Failure"
failed_mask = df[BINARY_TARGET] == 1
df.loc[failed_mask, MULTI_CLASS_TARGET] = (
    df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
    .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
)

print(f"✅ Data ready: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Failure rate: {df[BINARY_TARGET].mean() * 100:.2f}%")


## 2.1 — Distribution of each sensor (Histogram + KDE)

**What and why?**

We plot the distribution of each sensor to answer key questions:
- Is the distribution **normal/skewed**? (affects the choice of imputation and outlier handling)
- Are there **outliers** or strange clusters?

We use `histplot` with a density curve `kde=True` because it shows the shape of the distribution more smoothly than bars alone.


In [ ]:
def plot_distributions(df: pd.DataFrame, cols: list[str]) -> None:
    """
    Plot the distribution of each sensor (Histogram + KDE) in a subplot grid.

    Parameters
    ----------
    df : pd.DataFrame
        The data.
    cols : list[str]
        Names of the numeric sensor columns.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()

    for ax, col in zip(axes, cols):
        sns.histplot(df[col], kde=True, ax=ax, color="#4C72B0")
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("")
    # Hide the empty sixth axis
    axes[-1].set_visible(False)

    fig.suptitle("Sensor Reading Distributions", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_distributions(df, SENSOR_COLUMNS)


## 2.2 — Comparing readings between healthy and failed (Boxplot)

**What and why?**

This is the **most important** view of the problem: we place sensor readings side by side between two classes:
- `0` (healthy) and `1` (failed).

The boxplot reveals the **median, range, and outliers** of each class. If the failed class distribution is **clearly shifted** from the healthy class, that sensor **discriminates** failure — i.e. it will be a strong feature for the model.


In [ ]:
def plot_target_comparison(df: pd.DataFrame, cols: list[str]) -> None:
    """
    Boxplot comparison of each sensor between healthy and failed machines.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.ravel()

    for ax, col in zip(axes, cols):
        sns.boxplot(data=df, x=BINARY_TARGET, y=col, ax=ax,
                    hue=BINARY_TARGET, palette=["#4C72B0", "#C44E52"], legend=False)
        ax.set_title(col, fontsize=11)
        ax.set_xlabel("Machine failure (0 = Healthy, 1 = Failed)")

    axes[-1].set_visible(False)
    fig.suptitle("Sensor Readings: Healthy vs Failed", fontsize=14, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_target_comparison(df, SENSOR_COLUMNS)


## 2.3 — Correlation heatmap

**What and why?**

Correlation measures the **strength and direction** of the linear relationship between two variables (from -1 to +1):
- Close to **+1**: as one increases, the other increases.
- Close to **-1**: inverse relationship.
- Close to **0**: no linear relationship.

We add the target `Machine failure` to the matrix to see which sensors are **closest in correlation** to failure occurrence.

> ⚠️ Methodological note: correlation measures **linear** relationships only. Important non-linear relationships may exist that do not appear here — so we do not base decisions on this matrix alone.


In [ ]:
def plot_correlation(df: pd.DataFrame) -> None:
    """
    Plot the correlation matrix of the sensors + binary target.
    """
    corr_cols = SENSOR_COLUMNS + [BINARY_TARGET]
    corr = df[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
                center=0, vmin=-1, vmax=1, ax=ax,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    ax.set_title("Correlation: Sensors × Failure", fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()

plot_correlation(df)


## 2.4 — The categorical variable (Type): product quality

**What and why?**

`Type` represents product quality: `L` (Low), `M` (Medium), `H` (High). This is an important categorical variable because component quality physically affects the machine's resistance to failures.

We examine:
1. **Class distribution** — is it balanced?
2. **Failure rate within each class** — which class is most prone to failure?

This will confirm that `Type` is a feature that must be **encoded** (not dropped) in Phase 3.


In [ ]:
def analyze_type(df: pd.DataFrame) -> None:
    """
    Analyze the categorical 'Type' variable and its relationship with failure rate.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # (1) Class distribution
    counts = df[CATEGORICAL_COL].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette="viridis")
    axes[0].set_title("Product Quality Distribution (Type)")
    axes[0].set_xlabel("Quality")
    axes[0].set_ylabel("Count")

    # (2) Failure rate within each class
    failure_rate = df.groupby(CATEGORICAL_COL)[BINARY_TARGET].mean() * 100
    sns.barplot(x=failure_rate.index, y=failure_rate.values, ax=axes[1], palette="rocket")
    axes[1].set_title("Failure Rate (%) per Quality Class")
    axes[1].set_xlabel("Quality")
    axes[1].set_ylabel("Failure rate %")
    for i, v in enumerate(failure_rate.values):
        axes[1].text(i, v + 0.05, f"{v:.2f}%", ha="center", fontweight="bold")

    fig.tight_layout()
    plt.show()

    print("Failure rate per class:")
    print((df.groupby(CATEGORICAL_COL)[BINARY_TARGET].mean() * 100).round(2).to_string())

analyze_type(df)


## 2.5 — Relationship between each failure type and the sensors (physical analysis)

**What and why?**

Here we connect the dots with the **real physics** of the failures. From the original data paper we know the generative rules:

| Failure type | Physical generative rule |
|-----------|------------------------------|
| **TWF** (tool wear) | Tool wear exceeds a threshold (200–240 min) |
| **HDF** (heat dissipation) | Temp diff < 8.6K **and** speed < 1380 rpm |
| **PWF** (power) | Power = torque × speed < 3500W or > 9000W |
| **OSF** (overstrain) | Tool wear × torque > 11,000 min·Nm |
| **RNF** (random) | Completely random |

We plot the **mean tool wear** and **mean temperature difference** for each failure type, to visually verify that our data follows these rules — which strengthens our confidence in the correct understanding of the data before modeling.


In [ ]:
def analyze_failure_physics(df: pd.DataFrame) -> None:
    """
    Check data consistency against the physical rules of the failure types.
    """
    # Two physical features we examine against the failure types
    df = df.copy()
    df["Temp Diff [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # (1) Mean tool wear per failure type
    wear_by_type = df.groupby(MULTI_CLASS_TARGET)["Tool wear [min]"].mean()
    sns.barplot(x=wear_by_type.index, y=wear_by_type.values, ax=axes[0], palette="mako")
    axes[0].set_title("Mean Tool Wear per Failure Type")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].set_ylabel("Tool wear [min]")

    # (2) Mean temperature difference per failure type
    diff_by_type = df.groupby(MULTI_CLASS_TARGET)["Temp Diff [K]"].mean()
    sns.barplot(x=diff_by_type.index, y=diff_by_type.values, ax=axes[1], palette="flare")
    axes[1].set_title("Mean Temp Diff (Process - Air) per Failure Type")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_ylabel("Temp Diff [K]")

    fig.tight_layout()
    plt.show()

    print("Mean tool wear by type:")
    print(df.groupby(MULTI_CLASS_TARGET)["Tool wear [min]"].mean().round(1).to_string())

analyze_failure_physics(df)


## ✅ Phase 2 summary (results to remember before modeling)

From the exploration we extract **engineering insights** that will translate into decisions in Phase 3:

1. **Severe class imbalance (3.39%)** → we need `Stratified K-Fold` + `scale_pos_weight` and Recall/F1 evaluation.
2. **`Tool wear` and `Torque`** (and their physical derivatives) are the strongest discriminators of failure → we will build new engineered features from them.
3. **`Type` (quality)** clearly affects failure rate → it will be encoded with Ordinal (because L < M < H is a logical order).
4. **The physical rules of the failures** match the data → we can confidently engineer features like `Power = Torque × Speed` and `Temp Diff`.

### 🔜 Next: Phase 3 — Preprocessing & Feature Engineering + Pipeline
We will build `ColumnTransformer` + `Pipeline`, create the engineered features, and handle imbalance.
